## **Install Paddle**


In [ ]:
!pip install paddlepaddle==2.6.2 paddleocr==2.9.1

## **Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **NLTK for Tokenizer**

In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

# **Cek Duplikat**


In [ ]:
import cv2
import glob
import os
import difflib
import re
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI
# ==========================================
IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/cek_duplikat/*.*'
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

KATA_KUNCI_STRUK = ['total', 'tunai', 'kembali', 'rp', 'kasir', 'diskon', 'disc', 'voucher', 'harga', 'struk', 'bon', 'belanja']

def proses_ocr_dan_cek(img):
    """Mengeksekusi OCR, mengecek kelayakan teks, dan membuat sidik jari"""
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_prep = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    else:
        img_prep = img

    result = ocr_engine.ocr(img_prep, cls=True)
    if not result or result[0] is None:
        return False, "", "Teks sama sekali tidak terbaca"

    lines = [line[1][0].lower() for line in result[0]]
    full_text = " ".join(lines)

    raw_text = "".join([line[1][0] for line in result[0]])
    fingerprint = "".join(e for e in raw_text if e.isalnum()).lower()

    keyword_hits = sum(1 for kw in KATA_KUNCI_STRUK if kw in full_text)

    item_line_count = 0
    for text in lines:
        clean_text = text.replace('.', '').replace(',', '')
        numbers = re.findall(r'\d+', clean_text)
        if len(numbers) >= 2:
            has_qty = any(int(n) <= 100 for n in numbers if n.isdigit())
            has_price = any(int(n) >= 500 for n in numbers if n.isdigit())
            if has_qty and has_price:
                item_line_count += 1

    if len(lines) < 4:
        return False, fingerprint, f"Teks terlalu sedikit ({len(lines)} baris)"

    if item_line_count >= 1 and keyword_hits >= 1:
        return True, fingerprint, "Valid"

    return False, fingerprint, f"Kacau (Keyword: {keyword_hits}, Item: {item_line_count})"

def validasi_dan_ekstrak(img_path):
    """Mengatur alur Plan A dan Plan B, serta mengembalikan status untuk summary"""
    img = cv2.imread(img_path)
    if img is None:
        print("   -> [SKIP] Gambar tidak dapat dibuka.")
        return "", "Gambar rusak / tidak terbaca sistem"

    # ==========================================
    # PLAN A: Coba baca gambar asli
    # ==========================================
    is_valid, fingerprint, alasan = proses_ocr_dan_cek(img)
    if is_valid:
        print(f"   -> [LOLOS] Dimensi asli terbaca dengan baik.")
        return fingerprint, "Lolos"

    # ==========================================
    # PLAN B: Jika gagal & rasio 1:1, coba ditarik
    # ==========================================
    tinggi, lebar = img.shape[:2]
    if abs(tinggi - lebar) < (0.2 * lebar):
        print(f"   -> [INFO] Teks asli kacau & rasio 1:1. Mencoba stretch 1:2...")
        img_stretched = cv2.resize(img, (lebar, lebar * 2), interpolation=cv2.INTER_CUBIC)

        is_valid_s, fingerprint_s, alasan_s = proses_ocr_dan_cek(img_stretched)
        nama_file, ekstensi = os.path.splitext(img_path)

        if is_valid_s:
            path_baru = f"{nama_file}_stretched_succeed{ekstensi}"
            cv2.imwrite(path_baru, img_stretched)
            print(f"   -> [SUKSES] Teks berhasil terbaca! Disimpan sbg: _stretched_succeed")
            return fingerprint_s, "Lolos"
        else:
            path_baru = f"{nama_file}_stretched_failed{ekstensi}"
            cv2.imwrite(path_baru, img_stretched)
            print(f"   -> [GAGAL] Tetap kacau. Disimpan sbg: _stretched_failed")
            return "", f"Gagal Stretch (Plan A: {alasan} | Plan B: {alasan_s})"

    print(f"   -> [GAGAL] Dimensi bukan 1:1 dan teks asli kacau. Dilewati.")
    return "", f"Gagal Asli (Bukan 1:1) - {alasan}"


# ==========================================
# PROSES BATCH EKSTRAKSI
# ==========================================
print("[SISTEM] Memulai ekstraksi Sidik Jari Teks & Auto-Resize...\n")

image_paths = glob.glob(IMAGE_FOLDER)
valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

dokumen_teks = {}
nota_gagal = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)

    if "_stretched_" in filename:
        continue

    print(f"Membaca [{idx+1}/{len(image_paths)}]: {filename}")

    fingerprint, status = validasi_dan_ekstrak(path)
    if fingerprint:
        dokumen_teks[filename] = fingerprint
    else:
        nota_gagal.append((filename, status))

# ==========================================
# LAPORAN EVALUASI OCR
# ==========================================
print("\n" + "="*50)
print("LAPORAN EVALUASI OCR (FILE GAGAL/TIDAK TERBACA)")
print("="*50)
if nota_gagal:
    print(f"Ditemukan {len(nota_gagal)} nota yang tidak lolos filter OCR:\n")
    for file, alasan in nota_gagal:
        print(f" - {file}")
        print(f"   Detail: {alasan}\n")
else:
    print("Sempurna! Semua nota berhasil terbaca dan diekstrak dengan baik.")

# ==========================================
# FUZZY MATCHING (MENCARI DUPLIKAT)
# ==========================================
print("="*50)
print("LAPORAN DETEKSI DUPLIKASI NOTA (THRESHOLD 85%)")
print("="*50)

list_file = list(dokumen_teks.keys())
duplikat_ditemukan = False
sudah_dicek = set()

for i in range(len(list_file)):
    file1 = list_file[i]
    if file1 in sudah_dicek:
        continue

    teks1 = dokumen_teks[file1]
    grup_duplikat = [file1]

    for j in range(i + 1, len(list_file)):
        file2 = list_file[j]
        teks2 = dokumen_teks[file2]

        kemiripan = difflib.SequenceMatcher(None, teks1, teks2).ratio()

        if kemiripan >= 0.85:
            grup_duplikat.append(file2)
            sudah_dicek.add(file2)

    if len(grup_duplikat) > 1:
        duplikat_ditemukan = True
        print(f"\n[DUPLIKAT TERDETEKSI] Kemungkinan besar gambar ini adalah nota yang sama:")
        for nama in grup_duplikat:
            print(f" - {nama}")

if not duplikat_ditemukan:
    print("\nDataset bersih! Tidak ada duplikat pada file yang berhasil terbaca.")
print("\n" + "="*50)

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:16<00:00, 242.23it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:18<00:00, 537.93it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:15<00:00, 135.69it/s]


[SISTEM] Memulai ekstraksi Sidik Jari Teks & Auto-Resize...

Membaca [1/1049]: seteruk (368).jpg
   -> [INFO] Teks asli kacau & rasio 1:1. Mencoba stretch 1:2...
   -> [SUKSES] Teks berhasil terbaca! Disimpan sbg: _stretched_succeed
Membaca [2/1049]: seteruk (372).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [3/1049]: seteruk (337).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [4/1049]: seteruk (187).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [5/1049]: seteruk (333).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [6/1049]: seteruk (173).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [7/1049]: seteruk (19).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [8/1049]: seteruk (202).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [9/1049]: seteruk (345).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [10/1049]: seteruk (406).jpg
   -> [LOLOS] Dimensi asli terbaca dengan baik.
Membaca [11

# **Cek Kriteria Nota**

In [ ]:
#@title ngga resize

import cv2
import glob
import os
import re
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI
# ==========================================
IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/cek_nota/*.*'
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

KATA_KUNCI_STRUK = ['total', 'tunai', 'kembali', 'rp', 'kasir', 'diskon', 'disc', 'voucher', 'cash', 'harga', 'struk', 'bon', 'belanja']

def preprocess(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    return img

def validasi_struk(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return False, "Gambar korup / tidak bisa dibaca"

    img_prep = preprocess(img)
    result = ocr_engine.ocr(img_prep, cls=True)

    if not result or result[0] is None:
        return False, "Kosong (Mungkin sangat blur atau bukan teks)"

    lines = [line[1][0].lower() for line in result[0]]
    full_text = " ".join(lines)

    # 1. CEK KATA KUNCI (Minimal ada 1 kata kunci struk)
    keyword_hits = sum(1 for kw in KATA_KUNCI_STRUK if kw in full_text)

    # 2. CEK POLA RINCIAN BARANG (Qty & Harga)
    item_line_count = 0
    for text in lines:
        clean_text = text.replace('.', '').replace(',', '')
        numbers = re.findall(r'\d+', clean_text)

        if len(numbers) >= 2:
            has_qty = any(int(n) <= 100 for n in numbers if n.isdigit())
            has_price = any(int(n) >= 500 for n in numbers if n.isdigit())

            if has_qty and has_price:
                item_line_count += 1

    # ==========================================
    # LOGIKA PENGAMBILAN KEPUTUSAN
    # ==========================================
    if len(lines) < 4:
        return False, f"Teks terlalu sedikit ({len(lines)} baris)"

    if item_line_count >= 1 and keyword_hits >= 1:
        return True, "Lolos"

    alasan = f"Tidak ada pola rincian barang jelas. (Keyword: {keyword_hits}/1, Baris Item: {item_line_count}/1)"
    return False, alasan

# ==========================================
# EKSEKUSI BATCH
# ==========================================
print("[SISTEM] Memulai inspeksi kelayakan nota...\n")

image_paths = glob.glob(IMAGE_FOLDER)
valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

nota_lolos = 0
nota_mencurigakan = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)

    is_valid, alasan = validasi_struk(path)

    if is_valid:
        nota_lolos += 1
    else:
        nota_mencurigakan.append((filename, alasan))

    if (idx + 1) % 10 == 0:
        print(f"-> Memproses {idx+1}/{len(image_paths)} nota...")

# ==========================================
# LAPORAN AKHIR
# ==========================================
print("\n" + "="*50)
print("LAPORAN FILTER DATASET NOTA")
print("="*50)
print(f"Total Nota Diperiksa  : {len(image_paths)}")
print(f"Nota Lolos (Valid)    : {nota_lolos}")
print(f"Nota Mencurigakan     : {len(nota_mencurigakan)}")

if nota_mencurigakan:
    print("\n[DAFTAR NOTA MENCURIGAKAN - HARAP CEK MANUAL]:")
    for file, alasan in nota_mencurigakan:
        print(f" - {file} | Alasan: {alasan}")
else:
    print("\nLuar biasa! Seluruh notamu terdeteksi sebagai struk cetak mesin yang valid.")
print("="*50)

[SISTEM] Memulai inspeksi kelayakan nota...

-> Memproses 10/1123 nota...
-> Memproses 20/1123 nota...
-> Memproses 30/1123 nota...
-> Memproses 40/1123 nota...
-> Memproses 50/1123 nota...
-> Memproses 60/1123 nota...
-> Memproses 70/1123 nota...
-> Memproses 80/1123 nota...
-> Memproses 90/1123 nota...
-> Memproses 100/1123 nota...
-> Memproses 110/1123 nota...
-> Memproses 120/1123 nota...
-> Memproses 130/1123 nota...
-> Memproses 140/1123 nota...
-> Memproses 150/1123 nota...
-> Memproses 160/1123 nota...
-> Memproses 170/1123 nota...
-> Memproses 180/1123 nota...
-> Memproses 190/1123 nota...
-> Memproses 200/1123 nota...
-> Memproses 210/1123 nota...
-> Memproses 220/1123 nota...
-> Memproses 230/1123 nota...
-> Memproses 240/1123 nota...
-> Memproses 250/1123 nota...
-> Memproses 260/1123 nota...
-> Memproses 270/1123 nota...
-> Memproses 280/1123 nota...
-> Memproses 290/1123 nota...
-> Memproses 300/1123 nota...
-> Memproses 310/1123 nota...
-> Memproses 320/1123 nota...
-> M

In [ ]:
#@title resize 1:1 jadi 1:2 jika tidak terbaca

import cv2
import glob
import os
import re
from paddleocr import PaddleOCR

IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/cek_nota/persegi/*.*'
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
KATA_KUNCI_STRUK = ['total', 'tunai', 'kembali', 'rp', 'kasir', 'diskon', 'disc', 'voucher', 'cash', 'harga', 'struk', 'bon', 'belanja']

def stretch_image(img):
    """Fungsi khusus untuk menarik gambar kotak menjadi persegi panjang"""
    tinggi, lebar = img.shape[:2]
    tinggi_baru = lebar * 2
    return cv2.resize(img, (lebar, tinggi_baru), interpolation=cv2.INTER_CUBIC)

def proses_ocr_dan_cek(img):
    """Fungsi inti untuk mengekstrak dan mengecek syarat struk"""
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_prep = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    else:
        img_prep = img

    result = ocr_engine.ocr(img_prep, cls=True)
    if not result or result[0] is None:
        return False, "Teks kosong"

    lines = [line[1][0].lower() for line in result[0]]
    full_text = " ".join(lines)

    keyword_hits = sum(1 for kw in KATA_KUNCI_STRUK if kw in full_text)

    item_line_count = 0
    for text in lines:
        clean_text = text.replace('.', '').replace(',', '')
        numbers = re.findall(r'\d+', clean_text)
        if len(numbers) >= 2:
            has_qty = any(int(n) <= 100 for n in numbers if n.isdigit())
            has_price = any(int(n) >= 500 for n in numbers if n.isdigit())
            if has_qty and has_price:
                item_line_count += 1

    if len(lines) < 4:
        return False, f"Teks terlalu sedikit ({len(lines)} baris)"

    if item_line_count >= 1 and keyword_hits >= 1:
        return True, "Lolos"

    return False, f"Keyword: {keyword_hits}, Item: {item_line_count}"

def validasi_pintar(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return False, "Gambar korup"

    # ==========================================
    # PLAN A: Coba baca gambar asli dulu
    # ==========================================
    is_valid, alasan = proses_ocr_dan_cek(img)
    if is_valid:
        return True, "Lolos (Dimensi Asli)"

   # ==========================================
    # PLAN B: Jika gagal, dan gambarnya kotak, coba ditarik
    # ==========================================
    tinggi, lebar = img.shape[:2]
    if abs(tinggi - lebar) < (0.2 * lebar):
        img_stretched = stretch_image(img)

        nama_file, ekstensi = os.path.splitext(img_path)
        path_baru = f"{nama_file}_stretched{ekstensi}"
        cv2.imwrite(path_baru, img_stretched)

        is_valid_stretched, alasan_stretched = proses_ocr_dan_cek(img_stretched)

        if is_valid_stretched:
            return True, f"Lolos (Di-resize 1:2 & Disimpan sbg {os.path.basename(path_baru)})"

        alasan = f"Gagal dua kali. (Asli: {alasan} | Di-resize: {alasan_stretched})"

    return False, alasan

    return False, alasan

# ==========================================
# EKSEKUSI BATCH
# ==========================================
print("[SISTEM] Memulai inspeksi cerdas...\n")

image_paths = glob.glob(IMAGE_FOLDER)
valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

nota_mencurigakan = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)
    is_valid, status = validasi_pintar(path)

    if not is_valid:
        nota_mencurigakan.append((filename, status))

    if (idx + 1) % 10 == 0:
        print(f"-> Memproses {idx+1}/{len(image_paths)} nota...")

print("\n" + "="*50)
print(f"Total Nota Diperiksa  : {len(image_paths)}")
print(f"Nota Mencurigakan     : {len(nota_mencurigakan)}")

if nota_mencurigakan:
    print("\n[DAFTAR NOTA MENCURIGAKAN]:")
    for file, status in nota_mencurigakan:
        print(f" - {file} | Alasan: {status}")
print("="*50)

[SISTEM] Memulai inspeksi cerdas...

-> Memproses 10/14 nota...

Total Nota Diperiksa  : 14
Nota Mencurigakan     : 2

[DAFTAR NOTA MENCURIGAKAN]:
 - seteruk (647).jpg | Alasan: Gagal dua kali. (Asli: Teks kosong | Di-resize: Keyword: 3, Item: 0)
 - seteruk (87).jpg | Alasan: Gagal dua kali. (Asli: Keyword: 1, Item: 0 | Di-resize: Keyword: 2, Item: 0)


# **1st Trial - 200 nota awal**

## **OCR -> Doccano**

In [ ]:
#@title OCR Batch Processing

import cv2
import glob
import json
import os
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI PATH & ENGINE
# ==========================================
IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/1_100 nota/*.*'
OUTPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/import_doccano.jsonl'

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

# ==========================================
# FUNGSI PREPROCESSING & EKSTRAKSI
# ==========================================
def preprocess_for_doccano(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_for_ocr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return img_for_ocr
    return img

def extract_text(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[SKIP] Gagal membaca gambar: {img_path}")
        return ""

    img_prep = preprocess_for_doccano(img)

    result = ocr_engine.ocr(img_prep, cls=True)

    if not result or result[0] is None:
        return ""

    extracted_text = []
    for line in result[0]:
        text = line[1][0]
        extracted_text.append(text)

    return "\n".join(extracted_text)

# ==========================================
# PROSES BATCH
# ==========================================
print("[SISTEM] Memulai ekstraksi dataset untuk Doccano...\n")

image_paths = glob.glob(IMAGE_FOLDER)

valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

jsonl_data = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)
    print(f"Memproses {idx+1}/{len(image_paths)}: {filename}")

    text_result = extract_text(path)

    if text_result.strip():
        doccano_format = {"text": text_result}
        jsonl_data.append(doccano_format)

# ==========================================
# PENYIMPANAN KE FILE JSONL
# ==========================================
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

print("\n" + "="*50)
print(f"[SELESAI] Eksekusi tuntas!")
print(f"Total nota yang berhasil diekstrak: {len(jsonl_data)} dari {len(image_paths)}")
print(f"File hasil disimpan di: {OUTPUT_JSONL}")
print("="*50)

ModuleNotFoundError: No module named 'paddleocr'

##### **------------------**

In [ ]:
#@title OCR Batch Processing 2

import cv2
import glob
import json
import os
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI PATH & ENGINE
# ==========================================
IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/2_100 nota/*.*'
OUTPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/2_import_doccano.jsonl'

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

# ==========================================
# FUNGSI PREPROCESSING & EKSTRAKSI
# ==========================================
def preprocess_for_doccano(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_for_ocr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return img_for_ocr
    return img

def extract_text(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[SKIP] Gagal membaca gambar: {img_path}")
        return ""

    img_prep = preprocess_for_doccano(img)

    result = ocr_engine.ocr(img_prep, cls=True)

    if not result or result[0] is None:
        return ""

    extracted_text = []
    for line in result[0]:
        text = line[1][0]
        extracted_text.append(text)

    return "\n".join(extracted_text)

# ==========================================
# PROSES BATCH
# ==========================================
print("[SISTEM] Memulai ekstraksi dataset untuk Doccano...\n")

image_paths = glob.glob(IMAGE_FOLDER)

valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

jsonl_data = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)
    print(f"Memproses {idx+1}/{len(image_paths)}: {filename}")

    text_result = extract_text(path)

    if text_result.strip():
        doccano_format = {"text": text_result}
        jsonl_data.append(doccano_format)

# ==========================================
# PENYIMPANAN KE FILE JSONL
# ==========================================
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

print("\n" + "="*50)
print(f"[SELESAI] Eksekusi tuntas!")
print(f"Total nota yang berhasil diekstrak: {len(jsonl_data)} dari {len(image_paths)}")
print(f"File hasil disimpan di: {OUTPUT_JSONL}")
print("="*50)

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:22<00:00, 177.43it/s]


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:02<00:00, 4106.16it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:14<00:00, 142.60it/s]


[SISTEM] Memulai ekstraksi dataset untuk Doccano...

Memproses 1/100: nota_140.jpeg
Memproses 2/100: nota_127.jpeg
Memproses 3/100: nota_118.jpeg
Memproses 4/100: nota_138.jpeg
Memproses 5/100: nota_134.jpeg
Memproses 6/100: nota_147.jpeg
Memproses 7/100: nota_115.jpeg
Memproses 8/100: nota_120.jpeg
Memproses 9/100: nota_146.jpeg
Memproses 10/100: nota_129.jpeg
Memproses 11/100: nota_128.jpeg
Memproses 12/100: nota_130.jpeg
Memproses 13/100: nota_124.jpeg
Memproses 14/100: nota_123.jpeg
Memproses 15/100: nota_131.jpeg
Memproses 16/100: nota_122.jpeg
Memproses 17/100: nota_104.jpeg
Memproses 18/100: nota_105.jpeg
Memproses 19/100: nota_106.jpeg
Memproses 20/100: nota_107.jpeg
Memproses 21/100: nota_108.jpeg
Memproses 22/100: nota_109.jpeg
Memproses 23/100: nota_110.jpeg
Memproses 24/100: nota_111.jpeg
Memproses 25/100: nota_112.jpeg
Memproses 26/100: nota_113.jpeg
Memproses 27/100: nota_116.jpeg
Memproses 28/100: nota_117.jpeg
Memproses 29/100: nota_119.jpeg
Memproses 30/100: nota_121.j

In [ ]:
#@title OCR Batch Processing 2.1

import cv2
import glob
import json
import os
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI PATH & ENGINE
# ==========================================
IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/2_100 nota/nota_138.jpeg'
OUTPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/2-1_import_doccano_138.jsonl'

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

# ==========================================
# FUNGSI PREPROCESSING & EKSTRAKSI
# ==========================================
def preprocess_for_doccano(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_for_ocr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return img_for_ocr
    return img

def extract_text(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[SKIP] Gagal membaca gambar: {img_path}")
        return ""

    img_prep = preprocess_for_doccano(img)

    result = ocr_engine.ocr(img_prep, cls=True)

    if not result or result[0] is None:
        return ""

    extracted_text = []
    for line in result[0]:
        text = line[1][0]
        extracted_text.append(text)

    return "\n".join(extracted_text)

# ==========================================
# PROSES BATCH
# ==========================================
print("[SISTEM] Memulai ekstraksi dataset untuk Doccano...\n")

image_paths = glob.glob(IMAGE_FOLDER)

valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

jsonl_data = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)
    print(f"Memproses {idx+1}/{len(image_paths)}: {filename}")

    text_result = extract_text(path)

    if text_result.strip():
        doccano_format = {"text": text_result}
        jsonl_data.append(doccano_format)

# ==========================================
# PENYIMPANAN KE FILE JSONL
# ==========================================
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

print("\n" + "="*50)
print(f"[SELESAI] Eksekusi tuntas!")
print(f"Total nota yang berhasil diekstrak: {len(jsonl_data)} dari {len(image_paths)}")
print(f"File hasil disimpan di: {OUTPUT_JSONL}")
print("="*50)

[SISTEM] Memulai ekstraksi dataset untuk Doccano...

Memproses 1/1: nota_138.jpeg

[SELESAI] Eksekusi tuntas!
Total nota yang berhasil diekstrak: 1 dari 1
File hasil disimpan di: /content/drive/MyDrive/A.Skripshe/Dataset/2-1_import_doccano_138.jsonl


In [ ]:
#@title OCR Batch Processing 2.2

import cv2
import glob
import json
import os
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI PATH & ENGINE
# ==========================================
IMAGE_FOLDER = [
    '/content/drive/MyDrive/A.Skripshe/Dataset/2_100 nota/nota_206.jpeg',
    '/content/drive/MyDrive/A.Skripshe/Dataset/2_100 nota/nota_207.jpeg'
]
OUTPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/import_doccano_2-1_206-207.jsonl'

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

# ==========================================
# FUNGSI PREPROCESSING & EKSTRAKSI
# ==========================================
def preprocess_for_doccano(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_for_ocr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return img_for_ocr
    return img

def extract_text(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[SKIP] Gagal membaca gambar: {img_path}")
        return ""

    img_prep = preprocess_for_doccano(img)

    result = ocr_engine.ocr(img_prep, cls=True)

    if not result or result[0] is None:
        return ""

    extracted_text = []
    for line in result[0]:
        text = line[1][0]
        extracted_text.append(text)

    return "\n".join(extracted_text)

# ==========================================
# PROSES BATCH
# ==========================================
print("[SISTEM] Memulai ekstraksi dataset untuk Doccano...\n")

image_paths = IMAGE_FOLDER

valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

jsonl_data = []

for idx, path in enumerate(image_paths):
    filename = os.path.basename(path)
    print(f"Memproses {idx+1}/{len(image_paths)}: {filename}")

    text_result = extract_text(path)

    if text_result.strip():
        doccano_format = {"text": text_result}
        jsonl_data.append(doccano_format)

# ==========================================
# PENYIMPANAN KE FILE JSONL
# ==========================================
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

print("\n" + "="*50)
print(f"[SELESAI] Eksekusi tuntas!")
print(f"Total nota yang berhasil diekstrak: {len(jsonl_data)} dari {len(image_paths)}")
print(f"File hasil disimpan di: {OUTPUT_JSONL}")
print("="*50)

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:19<00:00, 202.66it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:02<00:00, 4549.45it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:17<00:00, 124.44it/s]


[SISTEM] Memulai ekstraksi dataset untuk Doccano...

Memproses 1/2: nota_206.jpeg
Memproses 2/2: nota_207.jpeg

[SELESAI] Eksekusi tuntas!
Total nota yang berhasil diekstrak: 2 dari 2
File hasil disimpan di: /content/drive/MyDrive/A.Skripshe/Dataset/import_doccano_2-1_206-207.jsonl


## **Doccano -> BIO**

In [ ]:
# 1. Mount Google Drive
import json
import nltk
from nltk.tokenize import WordPunctTokenizer

# ================= KONSUMEN CONFIG =================
INPUT_JSONL_PATH = '/content/drive/MyDrive/A.Skripshe/Dataset/admin.jsonl'
OUTPUT_BIO_PATH = '/content/drive/MyDrive/A.Skripshe/Dataset/adminBIO.txt'

def convert_doccano_to_bio(jsonl_path, output_path):
    tokenizer = WordPunctTokenizer()

    with open(jsonl_path, 'r', encoding='utf-8') as f_in, open(output_path, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            if not line.strip():
                continue

            data = json.loads(line)
            text = data['text']

            entities = data.get('label', data.get('entities', []))

            tokens_spans = list(tokenizer.span_tokenize(text))
            tokens = [text[start:end] for start, end in tokens_spans]

            bio_labels = ['O'] * len(tokens)

            for ent_start, ent_end, ent_label in entities:
                first_match = True
                for idx, (tok_start, tok_end) in enumerate(tokens_spans):
                    if tok_start >= ent_start and tok_end <= ent_end:
                        if first_match:
                            bio_labels[idx] = f"B-{ent_label}"
                            first_match = False
                        else:
                            bio_labels[idx] = f"I-{ent_label}"

            for token, label in zip(tokens, bio_labels):
                f_out.write(f"{token}\t{label}\n")

            f_out.write("\n")

    print(f"Konversi selesai! File BIO disimpan di: {output_path}")

convert_doccano_to_bio(INPUT_JSONL_PATH, OUTPUT_BIO_PATH)

Konversi selesai! File BIO disimpan di: /content/drive/MyDrive/A.Skripshe/Dataset/adminBIO.txt


# **2nd Trial - 1271 nota**

## **OCR - Docccano**

In [ ]:
#@title OCR Batch Processing

import cv2
import glob
import json
import os
import gc
import numpy as np
from sklearn.cluster import DBSCAN
from paddleocr import PaddleOCR

# ==========================================
# KONFIGURASI PATH & ENGINE
# ==========================================
IMAGE_FOLDER = '/content/drive/MyDrive/A.Skripshe/Dataset/Nota/*.*'
OUTPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/import_to_doccano.jsonl'

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

# ==========================================
# FUNGSI PREPROCESSING & EKSTRAKSI
# ==========================================
def preprocess_for_doccano(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_for_ocr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return img_for_ocr
    return img

def extract_text(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[SKIP] Gagal membaca gambar: {img_path}")
        return ""

    # ------------------------------------------
    # 1. RESIZE GAMBAR
    # ------------------------------------------
    tinggi, lebar = img.shape[:2]
    if lebar > batas_maksimal or tinggi > batas_maksimal:
        skala = batas_maksimal / max(tinggi, lebar)
        img = cv2.resize(img, (0, 0), fx=skala, fy=skala, interpolation=cv2.INTER_AREA)

    img_prep = preprocess_for_doccano(img)

    result = ocr_engine.ocr(img_prep, cls=True)

    if not result or result[0] is None:
        return ""

    # ------------------------------------------
    # 2. EKSTRAK BOUNDING BOX
    # ------------------------------------------
    boxes_and_texts = []
    for line in result[0]:
        box = line[0]
        text = line[1][0]

        y_min = min([point[1] for point in box])
        y_max = max([point[1] for point in box])
        x_min = min([point[0] for point in box])
        x_max = max([point[0] for point in box])

        y_center = (y_min + y_max) / 2
        height = y_max - y_min

        boxes_and_texts.append({
            'text': text,
            'y_center': y_center,
            'x_min': x_min,
            'x_max': x_max,
            'height': height
        })

    if not boxes_and_texts:
        return ""

    # ------------------------------------------
    # 3. DBSCAN CLUSTERING
    # ------------------------------------------
    y_centers = np.array([[item['y_center']] for item in boxes_and_texts])
    avg_height = np.mean([item['height'] for item in boxes_and_texts])

    clustering = DBSCAN(eps=avg_height * 0.35, min_samples=1).fit(y_centers)

    lines_dict = {}
    for label, item in zip(clustering.labels_, boxes_and_texts):
        if label not in lines_dict:
            lines_dict[label] = []
        lines_dict[label].append(item)

    lines = list(lines_dict.values())
    lines.sort(key=lambda line: np.mean([b['y_center'] for b in line]))

    # ------------------------------------------
    # 4. SPATIAL PADDING (Rongga Kolom)
    # ------------------------------------------
    final_text = []
    for line in lines:
        line.sort(key=lambda item: item['x_min'])

        line_text = ""
        for i, item in enumerate(line):
            if i == 0:
                line_text += item['text']
            else:
                prev_item = line[i-1]
                gap = item['x_min'] - prev_item['x_max']

                char_width = item['height'] * 0.5

                if gap > (char_width * 1.5):
                    num_spaces = int(gap / char_width)
                    num_spaces = min(max(num_spaces, 2), 10)
                    line_text += (" " * num_spaces) + item['text']
                else:
                    line_text += " " + item['text']

        final_text.append(line_text)

    del img, img_prep

    return "\n".join(final_text)

# ==========================================
# PROSES BATCH (DENGAN AUTO-RESUME & PELACAK GAGAL)
# ==========================================
print("[SISTEM] Memulai ekstraksi dataset untuk Doccano...\n")

image_paths = glob.glob(IMAGE_FOLDER)
valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = sorted([p for p in image_paths if p.lower().endswith(valid_extensions)])

TRACKER_FILE = '/content/drive/MyDrive/A.Skripshe/Dataset/tracker_ocr_doccano.txt'
GAGAL_FILE = '/content/drive/MyDrive/A.Skripshe/Dataset/nota_gagal_ekstrak.txt'

sudah_diproses = set()
if os.path.exists(TRACKER_FILE):
    with open(TRACKER_FILE, 'r', encoding='utf-8') as f:
        sudah_diproses = set(f.read().splitlines())

print(f"-> Menemukan {len(sudah_diproses)} nota yang sudah diproses sebelumnya.")

nota_gagal = [] #

with open(OUTPUT_JSONL, 'a', encoding='utf-8') as f_json, open(TRACKER_FILE, 'a', encoding='utf-8') as f_track:

    for idx, path in enumerate(image_paths):
        filename = os.path.basename(path)

        if filename in sudah_diproses:
            continue

        print(f"Memproses [{idx+1}/{len(image_paths)}]: {filename}")
        text_result = extract_text(path)

        if text_result.strip():
            doccano_format = {"text": text_result}
            f_json.write(json.dumps(doccano_format) + '\n')
            f_json.flush()
        else:
            nota_gagal.append(filename)

        f_track.write(filename + '\n')
        f_track.flush()

        gc.collect()

if nota_gagal:
    with open(GAGAL_FILE, 'a', encoding='utf-8') as f_gagal:
        for nama in nota_gagal:
            f_gagal.write(nama + '\n')

print("\n" + "="*50)
print(f"[SELESAI] Eksekusi tuntas sampai akhir!")
print(f"File Doccano   : {OUTPUT_JSONL}")
print(f"File Tracker   : {TRACKER_FILE}")
if nota_gagal:
    print(f"Ada {len(nota_gagal)} nota yang GAGAL/KOSONG. Cek di: {GAGAL_FILE}")
print("="*50)

[SISTEM] Memulai ekstraksi dataset untuk Doccano...

-> Menemukan 236 nota yang sudah diproses sebelumnya.
Memproses [237/1271]: nota_0237.jpg
Memproses [238/1271]: nota_0238.jpg
Memproses [239/1271]: nota_0239.jpg
Memproses [240/1271]: nota_0240.jpg
Memproses [241/1271]: nota_0241.jpg
Memproses [242/1271]: nota_0242.jpg
Memproses [243/1271]: nota_0243.jpg
Memproses [244/1271]: nota_0244.jpg
Memproses [245/1271]: nota_0245.jpg
Memproses [246/1271]: nota_0246.jpg
Memproses [247/1271]: nota_0247.jpg
Memproses [248/1271]: nota_0248.jpg
Memproses [249/1271]: nota_0249.jpg
Memproses [250/1271]: nota_0250.jpg
Memproses [251/1271]: nota_0251.jpg
Memproses [252/1271]: nota_0252.jpg
Memproses [253/1271]: nota_0253.jpg
Memproses [254/1271]: nota_0254.jpg
Memproses [255/1271]: nota_0255.jpg
Memproses [256/1271]: nota_0256.jpg
Memproses [257/1271]: nota_0257.jpg
Memproses [258/1271]: nota_0258.jpg
Memproses [259/1271]: nota_0259.jpg
Memproses [260/1271]: nota_0260.jpg
Memproses [261/1271]: nota_02

In [ ]:
#@title diberi regex untuk pisahin huruf angka dan angka angka

import json
import re

INPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/import_to_doccano.jsonl'
OUTPUT_JSONL = '/content/drive/MyDrive/A.Skripshe/Dataset/import_doccano_all.jsonl'

with open(INPUT_JSONL, 'r', encoding='utf-8') as f_in, \
     open(OUTPUT_JSONL, 'w', encoding='utf-8') as f_out:

    for line in f_in:
        data = json.loads(line)
        teks = data['text']

        # 1. Pisahkan Huruf ketemu Angka (Contoh: Total15000 -> Total 15000)
        teks = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', teks)

        # 2. Pisahkan Angka ketemu Huruf (Contoh: 15000Rp -> 15000 Rp)
        teks = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', teks)

        # 3. Pisahkan Tanggal (Tahun) ketemu Jam (Contoh: 202315:30 -> 2023 15:30)
        teks = re.sub(r'(\d{2,4})(\d{2}[.:]\d{2})', r'\1 \2', teks)


        # 4. Pisahkan Tanggal dari noise di depan atau belakangnya

        # Dedine pola Tanggal Angka dan Huruf
        # Pola 1: Angka Full (Contoh: 15/12/2023, 15-12-23)
        pola_tgl_angka = r'(\d{2}[-/.]\d{2}[-/.](?:20\d{2}|(?!20)\d{2}))'

        # Pola 2: Angka + Huruf Bulan (Contoh: 15 Jan 2023, 15Jan23)
        pola_tgl_huruf = r'(\d{2}[-/.\s]*[a-zA-Z]+[-/.\s]*(?:20\d{2}|(?!20)\d{2}))'

        # Depan
        teks = re.sub(r'(\d+)' + pola_tgl_angka, r'\1 \2', teks)
        teks = re.sub(r'(\d+)' + pola_tgl_huruf, r'\1 \2', teks)

        # Belakang
        teks = re.sub(pola_tgl_angka + r'(\d+)', r'\1 \2', teks)
        teks = re.sub(pola_tgl_huruf + r'(\d+)', r'\1 \2', teks)

        data['text'] = teks
        f_out.write(json.dumps(data) + '\n')

print("Selesai! Gunakan 'import_doccano_fixed.jsonl' untuk diimpor ke Doccano.")

Selesai! Gunakan 'import_doccano_fixed.jsonl' untuk diimpor ke Doccano.


## **Doccano BIo**

In [ ]:
# 1. Mount Google Drive
import json
import nltk
from nltk.tokenize import WordPunctTokenizer

# ================= KONSUMEN CONFIG =================
INPUT_JSONL_PATH = '/content/drive/MyDrive/A.Skripshe/Dataset/doccano.jsonl'
OUTPUT_BIO_PATH = '/content/drive/MyDrive/A.Skripshe/Dataset/doccanoBIO.txt'

def convert_doccano_to_bio(jsonl_path, output_path):
    tokenizer = WordPunctTokenizer()

    with open(jsonl_path, 'r', encoding='utf-8') as f_in, open(output_path, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            if not line.strip():
                continue

            data = json.loads(line)
            text = data['text']

            entities = data.get('label', data.get('entities', []))

            tokens_spans = list(tokenizer.span_tokenize(text))
            tokens = [text[start:end] for start, end in tokens_spans]

            bio_labels = ['O'] * len(tokens)

            for ent_start, ent_end, ent_label in entities:
                first_match = True
                for idx, (tok_start, tok_end) in enumerate(tokens_spans):
                    if tok_start >= ent_start and tok_end <= ent_end:
                        if first_match:
                            bio_labels[idx] = f"B-{ent_label}"
                            first_match = False
                        else:
                            bio_labels[idx] = f"I-{ent_label}"

            for token, label in zip(tokens, bio_labels):
                f_out.write(f"{token}\t{label}\n")

            f_out.write("\n")

    print(f"Konversi selesai! File BIO disimpan di: {output_path}")

convert_doccano_to_bio(INPUT_JSONL_PATH, OUTPUT_BIO_PATH)

Konversi selesai! File BIO disimpan di: /content/drive/MyDrive/A.Skripshe/Dataset/doccanoBIO.txt


In [ ]:
#@title script penyembuh label BIO

input_bio = '/content/drive/MyDrive/A.Skripshe/Dataset/doccanoBIO.txt'
output_bio_bersih = '/content/drive/MyDrive/A.Skripshe/Dataset/doccanoBIO_bersih.txt'

def sembuhkan_bio(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8') as f_out:
        prev_tag = "O"

        for line in f_in:
            line = line.strip()
            if not line:
                f_out.write("\n")
                prev_tag = "O"
                continue

            parts = line.split('\t')
            if len(parts) == 2:
                token, current_tag = parts

                if current_tag.startswith("I-"):
                    entity_type = current_tag[2:]
                    prev_entity_type = prev_tag[2:] if prev_tag != "O" else ""

                    if prev_tag == "O" or entity_type != prev_entity_type:
                        current_tag = f"B-{entity_type}"
                # ------------------------------

                f_out.write(f"{token}\t{current_tag}\n")
                prev_tag = current_tag
            else:
                f_out.write(line + "\n")

print("Memulai pembersihan aturan BIO...")
sembuhkan_bio(input_bio, output_bio_bersih)
print("Selesai! Gunakan 'doccanoBIO_bersih.txt' untuk proses latih AI.")

Memulai pembersihan aturan BIO...
Selesai! Gunakan 'doccanoBIO_bersih.txt' untuk proses latih AI.
